In [ ]:

import requests
import pandas as pd

# 1. Captura o parâmetro dinâmico do Bundle usando widgets do Databricks
dbutils.widgets.text("bundle_target", "dev") # Define 'dev' como valor fallback
ambiente = dbutils.widgets.get("bundle_target")

print(f"=========================================")
print(f"Iniciando Extração de API no ambiente: {ambiente.upper()}")
print(f"=========================================")

# 2. Faz a requisição na API pública (trazendo 50 registros)
url = "https://randomuser.me/api/?results=50"
resposta = requests.get(url)
dados_json = resposta.json()["results"]

# 3. Trata o JSON achatando as estruturas aninhadas com Pandas
df_pandas = pd.json_normalize(dados_json)

# Seleciona algumas colunas para deixar a tabela limpa
df_pandas = df_pandas[['name.first', 'name.last', 'email', 'location.country']]

# 4. Converte para Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)
display(df_spark)

# 5. Salva como Tabela Delta no Unity Catalog
# O nome da tabela muda dinamicamente: ex -> default.dev_usuarios_api
nome_tabela = f"default.{ambiente}_usuarios_api"

print(f"Salvando dados na tabela Delta: {nome_tabela}...")
df_spark.write.mode("overwrite").saveAsTable(nome_tabela)

print("Processo finalizado com sucesso!")